CATALOGO MENOR (UMA PLANILHA PARA TODAS AS PASTAS)

In [1]:
#CATALOGO MENOR

import os
import re
from PIL import Image
import pytesseract
import csv

# 1. Pasta raiz com todas as subpastas
pasta_raiz = "./BARRA E FERRAZ"
# 2. Categoria (fixa)
categoria = "05 09 25"

# Configuração do caminho do Tesseract
pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

extensoes = ('.png', '.jpg', '.jpeg')
resultados = []

# Função para processar o texto extraído e estruturar os dados
def processar_texto(texto, arquivo, Nmodelo):
    linhas = texto.strip().split("\n")
    linhas = [linha.strip() for linha in linhas if linha.strip()]

    if len(linhas) >= 2:
        nomeIDcompleto = linhas[0]
        preco = linhas[1]
    else:
        nomeIDcompleto = "Desconhecido"
        preco = "0.00"

    preco = preco.replace("R$", "").strip()
    preco = re.sub(r"[^0-9,]", "", preco)

    if not preco:
        preco = "0,00"

    if "." in nomeIDcompleto:
        partes = nomeIDcompleto.split(".")
        nomeID = partes[0]
        tamanho = partes[1] if partes[1].isdigit() else ""
    else:
        nomeID = nomeIDcompleto
        tamanho = ""

    if nomeID.endswith("01"):
        Ntipo = ""
    elif nomeID.endswith("66"):
        Ntipo = "DISPONIVEL NA PRATA 925"
    elif nomeID.endswith("73"):
        Ntipo = "DISPONIVEL NO RODIO"
    elif nomeID.endswith("45"):
        Ntipo = ""
    elif nomeID.endswith("65"):
        Ntipo = ""
    else:
        Ntipo = "N/A"

    nome = f"{Nmodelo} {Ntipo}".strip()
    sku = os.path.splitext(arquivo)[0]

    return {
        "Nome": nome,
        "Categoria": categoria,
        "Variacao": tamanho,
        "SKU": sku,
        "Preco": preco,
        "codigo": nomeID
    }

# Percorrer todas as subpastas dentro da pasta raiz
for subpasta in os.listdir(pasta_raiz):
    caminho_subpasta = os.path.join(pasta_raiz, subpasta)

    # Verifica se é uma pasta
    if os.path.isdir(caminho_subpasta):
        Nmodelo = subpasta  # Nome da pasta vira o modelo

        # Processar todas as imagens dentro da subpasta
        for arquivo in os.listdir(caminho_subpasta):
            if arquivo.lower().endswith(extensoes):
                img = os.path.join(caminho_subpasta, arquivo)
                texto = pytesseract.image_to_string(img, lang="eng")

                dados_formatados = processar_texto(texto, arquivo, Nmodelo)
                resultados.append(dados_formatados)

# Salvar todos os resultados em um CSV único
csv_path = "resultadosT.csv"
with open(csv_path, mode='w', newline='', encoding='utf-8') as file:
    writer = csv.DictWriter(file, fieldnames=["Nome", "Categoria", "SKU", "Variacao", "Preco", "codigo"], delimiter=';')
    writer.writeheader()
    writer.writerows(resultados)

print(f"Resultados salvos em: {csv_path}")


Resultados salvos em: resultadosT.csv


CATALOGO GRANDE (UMA PLANILHA PARA CADA PASTA)

In [3]:
#CATALOGO GRANDE

import os
import re
import csv
import easyocr
import logging


#pip install pillow
#pip install easyocr

# Desativar logs desnecessários
logging.getLogger('easyocr').setLevel(logging.ERROR)

# 0. Tirar todos os _02.jpg

# 1. Pasta raiz com todas as subpastas
pasta_raiz = "./BARRA E FERRAZ"
# 2. Pasta de saída para os CSVs
pasta_saida = "./planilhas"
# 3. Sufixo fixo do lote/data para a categoria
sufixo_categoria = "05 04 26"

extensoes = ('.png', '.jpg', '.jpeg')

# Criar a pasta de saída se não existir
os.makedirs(pasta_saida, exist_ok=True)

# --- INICIALIZAÇÃO DO EASYOCR ---
# lang='pt' ativa o dicionário de português para R$ e acentos
print("Inicializando EasyOCR... (Na primeira vez isso pode demorar um pouco)")
reader = easyocr.Reader(['pt', 'en'])


# Função para processar o texto extraído e estruturar os dados
def processar_texto(texto_bruto, arquivo, Nmodelo):
    linhas = [linha.strip() for linha in texto_bruto if linha.strip()]
    
    # Garante que não quebre se a lista for vazia
    if len(linhas) >= 2:
        nomeIDcompleto = linhas[0]
        preco = linhas[1]
    elif len(linhas) == 1:
        nomeIDcompleto = linhas[0]
        preco = "0,00"
    else:
        nomeIDcompleto = "Desconhecido"
        preco = "0,00"

    preco = preco.replace("R$", "").strip()
    preco = re.sub(r"[^0-9,]", "", preco)
    if not preco: preco = "0,00"

    if "." in nomeIDcompleto:
        partes = nomeIDcompleto.split(".")
        nomeID = partes[0]
        variacao = partes[-1]
        tamanho = variacao if variacao.isdigit() else ""
    else:
        nomeID = nomeIDcompleto
        tamanho = ""

    # Lógica de sufixos (Branco para 01, 45, 65 | N/A para o resto)
    if nomeID.endswith("01") or nomeID.endswith("45") or nomeID.endswith("65"):
        Ntipo = ""
    elif nomeID.endswith("66"):
        Ntipo = "DISPONIVEL NA PRATA 925"
    elif nomeID.endswith("73"):
        Ntipo = "DISPONIVEL NO RODIO"
    else:
        Ntipo = "N/A"

    categoria_dinamica = f"{Nmodelo} {sufixo_categoria}"

    return {
        "Nome": f"{Nmodelo} {Ntipo}".strip(),
        "Categoria": categoria_dinamica,
        "Variacao": tamanho,
        "SKU": os.path.splitext(arquivo)[0],
        "Preco": preco,
        "codigo": nomeID
    }

# --- EXECUÇÃO ---
pasta_raiz = "./BARRA E FERRAZ"
pasta_saida = "./planilhas"
os.makedirs(pasta_saida, exist_ok=True)

for subpasta in os.listdir(pasta_raiz):
    caminho_subpasta = os.path.join(pasta_raiz, subpasta)
    if os.path.isdir(caminho_subpasta):
        Nmodelo = subpasta
        resultados = []
        print(f"Processando pasta: {Nmodelo}")

        for arquivo in os.listdir(caminho_subpasta):
            if arquivo.lower().endswith(('.png', '.jpg', '.jpeg')):
                img_path = os.path.join(caminho_subpasta, arquivo)
                
                try:
                    # Na 3.x use apenas o predict ou ocr sem argumentos extras
                    texto_extraido = reader.readtext(img_path, detail=0)
                    
                    dados = processar_texto(texto_extraido, arquivo, Nmodelo)
                    resultados.append(dados)
                    print(f"  [OK] {arquivo}")
                except Exception as e:
                    print(f"  [ERRO] {arquivo}: {e}")

        # Salvar CSV
        if resultados:
            csv_path = os.path.join(pasta_saida, f"{Nmodelo}.csv")
            with open(csv_path, mode='w', newline='', encoding='utf-8-sig') as f:
                writer = csv.DictWriter(f, fieldnames=["Nome", "Categoria", "SKU", "Variacao", "Preco", "codigo"], delimiter=';')
                writer.writeheader()
                writer.writerows(resultados)

ImportError: cannot import name 'package' from partially initialized module 'torch' (most likely due to a circular import) (C:\Users\rafaf\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torch\__init__.py)